# 01 - AOI & boundaries (Phase 1)

**Colombo UHI practicum.** Builds and visualises every study-area geometry:
Colombo District + Western Province (FAO GAUL), DS/GN divisions (user asset,
with fallback), the CMC boundary (~37 km2 sanity check), the GHSL-derived
urban extent, the combined water mask, and BOTH SUHII rural-reference
definitions (`buffer_ring`, `lcz_based`).

Run top-to-bottom in **Google Colab** after `00_setup_and_auth.ipynb` has
worked once. All logic lives in `src/colombo_uhi/aoi.py`; this notebook only
orchestrates and displays.

> **Caveat (CLAUDE.md #1):** everything in this project is LAND SURFACE
> TEMPERATURE analysis - never air temperature. The exact caveat string is
> printed from `params["caveats"]` below and must accompany every product.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))
from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
print("CAVEAT:", params["caveats"]["lst_not_air_temp"])

In [ ]:
# COLAB: RUN THIS CELL
# Administrative boundaries. Expect a PROMINENT warning for DS/GN (assets are
# still null in params.yaml) and an actionable error message for the CMC.
from colombo_uhi import aoi

district = aoi.colombo_district(params)
province = aoi.western_province(params)
print("District features (expect 1):", district.size().getInfo())
print("Province features (expect 3):", province.size().getInfo())
print("Province districts:",
      province.aggregate_array(params["aoi"]["gaul"]["district_property"]).getInfo())

ds_fc = aoi.ds_divisions(params)   # falls back to GAUL district while asset is null
gn_fc = aoi.gn_divisions(params)   # ditto
print("\nDS features (expect 13 once the asset is uploaded):", ds_fc.size().getInfo())
print("GN features (expect 557 once the asset is uploaded):", gn_fc.size().getInfo())

cmc_geom = None
try:
    cmc_geom = aoi.cmc_boundary(params)
except RuntimeError as err:
    print("\nCMC boundary not available yet -", err)

In [ ]:
# COLAB: RUN THIS CELL
# Area sanity checks. GAUL is simplified at 500 m - a few % deviation is normal.
# The urban-extent vectorisation makes this cell take ~1 minute.
expected = params["aoi"]["expected_areas_km2"]

urban_geom = aoi.urban_extent(params)
ring_geom = aoi.buffer_ring(params)

rows = [
    ("Colombo District", district.geometry(10), expected["district"]),
    ("Western Province", province.geometry(10), expected["western_province"]),
    ("CMC", cmc_geom, expected["cmc"]),
    ("Urban extent (GHSL)", urban_geom, None),
    ("Rural buffer ring", ring_geom, None),
]
print(f"{'AOI':<22}{'area km2':>12}{'expected':>12}")
for name, geom, exp in rows:
    if geom is None:
        print(f"{name:<22}{'- (asset missing)':>12}{exp or '':>12}")
        continue
    km2 = aoi.area_km2(geom).getInfo()
    flag = ""
    if exp:
        flag = "  OK" if abs(km2 - exp) / exp <= 0.10 else "  << CHECK (>10% off)"
    print(f"{name:<22}{km2:>12.1f}{exp or '':>12}{flag}")

In [ ]:
# COLAB: RUN THIS CELL
# Combined water mask: MNDWI OR QA_PIXEL-water-frequency OR JRC occurrence.
# Plus a 60 m shoreline-buffer variant (coastal mixed-pixel exclusion demo;
# the project default aoi.water_mask.shoreline_buffer_m is 0 = off).
water = aoi.water_mask(params)
water_buffered = aoi.water_exclusion_mask(params, shoreline_buffer_m=60)
print("Water mask built. Threshold config:", params["aoi"]["water_mask"])

In [ ]:
# COLAB: RUN THIS CELL
# BOTH SUHII rural-reference definitions behind the common interface.
# Later phases flip between them with this one string (CLAUDE.md caveat 5:
# always report the sensitivity, never a single number).
urban_br, rural_br = aoi.rural_reference("buffer_ring", params)
urban_lcz, rural_lcz = aoi.rural_reference("lcz_based", params)
print("buffer_ring: ring", params["uhi"]["suhii"]["buffer_ring"]["inner_km"], "-",
      params["uhi"]["suhii"]["buffer_ring"]["outer_km"], "km beyond the",
      params["uhi"]["suhii"]["buffer_ring"]["base"], "; excludes",
      params["uhi"]["suhii"]["buffer_ring"]["exclude"])
print("lcz_based: urban =", params["uhi"]["suhii"]["lcz_based"]["urban_classes"],
      "| rural =", params["uhi"]["suhii"]["lcz_based"]["rural_classes"],
      "(A-G; water/class G removed by the water mask)")

In [ ]:
# COLAB: RUN THIS CELL
# Interactive map of everything. Toggle layers in the layer control.
import ee
import geemap

centre = params["aoi"]["centre"]
m = geemap.Map(center=[centre["lat"], centre["lon"]], zoom=9)

def outline(fc_or_geom, color, width=2):
    fc = fc_or_geom
    if isinstance(fc_or_geom, ee.Geometry):
        fc = ee.FeatureCollection([ee.Feature(fc_or_geom)])
    return fc.style(color=color, fillColor="00000000", width=width)

m.addLayer(outline(province, "000000"), {}, "Western Province (GAUL)")
m.addLayer(outline(district, "d62728"), {}, "Colombo District (GAUL)")
if cmc_geom is not None:
    m.addLayer(outline(cmc_geom, "9467bd", 3), {}, "CMC (from DS asset)")
m.addLayer(outline(urban_geom, "ff7f0e"), {}, "Urban extent (GHSL)")
m.addLayer(outline(ring_geom, "2ca02c"), {}, "Rural buffer ring (15-25 km)")

m.addLayer(water.selfMask(), {"palette": ["1f77b4"]}, "Water mask")
m.addLayer(water_buffered.selfMask(), {"palette": ["17becf"]},
           "Water mask + 60 m shoreline buffer", False)
m.addLayer(rural_br.selfMask(), {"palette": ["98df8a"]},
           "Rural mask - buffer_ring method", False)
m.addLayer(urban_lcz.selfMask(), {"palette": ["7f7f7f"]},
           "Urban mask - LCZ classes 1-10", False)
m.addLayer(rural_lcz.selfMask(), {"palette": ["2ca02c"]},
           "Rural mask - LCZ A-G minus water", False)
m

## Uploading the DS/GN boundary assets (removes the fallback warnings)

GAUL stops at district level for Sri Lanka, so DS divisions, GN divisions and
the CMC boundary need a **user-uploaded EE asset** (CLAUDE.md anticipated this
for GN; it applies to DS/CMC too - see PROGRESS.md, 2026-08-08).

1. Download **"Sri Lanka - Subnational Administrative Boundaries"** from
   OCHA/HDX (<https://data.humdata.org/dataset/cod-ab-lka>):
   the **admin3** shapefile = DS divisions, **admin4** = GN divisions.
   Check the dataset's licence page before redistributing derived maps.
2. In the EE Code Editor (<https://code.earthengine.google.com>):
   *Assets > New > Shape files* - upload each shapefile set
   (`.shp .shx .dbf .prj` together). Suggested ids:
   `projects/research-uhi-484404/assets/lka_ds_divisions` and
   `.../lka_gn_divisions`.
3. Put those asset ids into `config/params.yaml` under
   `aoi.assets.ds_divisions` / `aoi.assets.gn_divisions`, commit, push,
   and re-run this notebook.
4. Verify: DS count filtered to Colombo District = **13**, GN count = **557**,
   CMC area within a few % of **37 km2**. If the CMC area prints ~0, the name
   property differs - inspect the asset's attribute table and adjust
   `aoi.cmc.ds_name_property` (default `ADM3_EN`).
   Note: the uploaded layers cover ALL of Sri Lanka; the counts above apply
   after filtering to Colombo District (district-name attribute, e.g. `ADM2_EN`).

## Visual verification checklist (Phase 1 sign-off)

- [ ] District = exactly **1** feature; area within ~10% of **699 km2**.
- [ ] Western Province = **3** features (Colombo, Gampaha, Kalutara);
      combined area near **3684 km2**.
- [ ] DS/GN fallback warning printed prominently (until assets are uploaded).
- [ ] CMC error message is clear and actionable (until the DS asset exists);
      afterwards CMC area ~ **37 km2**.
- [ ] Water mask covers: Indian Ocean, **Beira Lake**, **Bolgoda Lake**,
      **Kelani River**, **Diyawanna (Parliament) Lake**. Zoom to each.
- [ ] The 60 m shoreline-buffer variant is visibly fatter along the coastline.
- [ ] Urban extent (orange) hugs the Colombo built-up area; the rural ring
      (green) sits 15-25 km outside it; ring pixels over ocean/water are
      removed in the *rural mask* layer (toggle it on).
- [ ] LCZ urban mask (grey) sits over central Colombo; LCZ rural mask (green)
      contains no water pixels.

**Next:** Phase 2 - `02_lst_pipeline.ipynb` (harmonised Landsat LST + MODIS).